# Anomaly detection on sensor time series

Five detectors on nine labeled series from the [Numenta Anomaly Benchmark](https://github.com/numenta/NAB):
rolling z-score (baseline), Isolation Forest on window features, matrix profile discords (stumpy, z-normalized and raw)
and a 1D-conv autoencoder.
The sixth entry, `ensemble`, averages the validation-calibrated scores of the raw matrix profile and the z-score at twice the context length; the leave-one-out optimization at the end selected this combination.

Each series is split chronologically into train (40%), validation (20%) and test (40%). Detectors are fit on train with labeled
windows removed. For series with a weekly cycle (nyc_taxi, ambient_temperature) the median time-of-week profile, estimated on
normal training points, is subtracted first. The threshold is the same score quantile on normal validation points for every detector.

Evaluation runs on test at event level: a labeled window counts as detected if at least one alarm falls inside it, each alarm run
outside a window is one false positive. Alarms within one context length after a window are ignored, because the trailing window
still contains the anomaly. Isolation Forest and autoencoder are run with five seeds.

The second part repeats the comparison on the multivariate [SKAB](https://github.com/waico/SKAB) benchmark, explains which sensor
drives each alarm and shows the detector running point by point.

Interactive version with a threshold slider: [Streamlit app](https://anomalydetectionsensor-53njzwcxxrjku7qqqrjpns.streamlit.app/).

In [ ]:
import json

import numpy as np
import pandas as pd
import plotly.io as pio

from src.data import ROOT
from src.plots import attribution_heatmap, hero, small_multiples, sweep_chart
from src.skab import SENSORS

pio.renderers.default = "notebook_connected"
RESULTS = ROOT / "results"
metrics = pd.read_csv(RESULTS / "metrics.csv")
sweep = pd.read_csv(RESULTS / "sweep.csv")
scores = pd.read_parquet(RESULTS / "scores.parquet")
meta = json.loads((RESULTS / "meta.json").read_text())
order = list(metrics["detector"].unique())

## Results

NAB windows are wide, so random alarms often land inside them. The chance F1 shifts each detector's test alarm sequence by a random
circular offset (1000 times), which keeps the number and length of alarm runs but breaks their alignment with the labels.
The p-value is the share of shifts reaching at least the observed F1.

Cells show event F1 (mean over seeds) with chance F1 in brackets; `*` marks p < 0.05.

In [ ]:
per = metrics.groupby(["series", "detector"], sort=False).agg(
    f1=("f1", "mean"), chance=("chance_f1", "mean"), p=("p_value", "median"), events=("events", "first"),
)
cell = per.apply(lambda r: f"{r.f1:.2f} ({r.chance:.2f}){'*' if r.p < 0.05 else ''}", axis=1)
table = cell.unstack("detector")[order]
table.insert(0, "test events", per["events"].unstack("detector").iloc[:, 0])
table

In [ ]:
seed_sd = metrics.groupby(["series", "detector"])["f1"].std().groupby("detector").mean()
summary = per.groupby("detector").agg(
    f1=("f1", "mean"), chance=("chance", "mean"), significant=("p", lambda p: f"{(p < 0.05).sum()}/{len(p)}"),
)
summary["f1 − chance"] = summary["f1"] - summary["chance"]
summary["seed sd"] = seed_sd
summary = summary.join(metrics.groupby("detector")[["precision", "recall", "delay_h", "point_f1"]].mean())
summary.loc[order].round(2)

Delay is measured from the labeled anomaly point, which NAB places in the middle of each window, so values between minus and plus
half a window width are expected even for alarms without real lead time.

## Threshold sensitivity

Event F1 for a range of validation quantiles. A detector whose curve stays high over several quantiles ranks anomalies well;
a sharp peak means the result depends on the threshold choice. The best quantile is chosen on test data here and is therefore optimistic.

In [ ]:
sweep_chart(sweep)

## Detections over time

Grey bands are the labeled anomaly windows, the dotted line marks the start of the test part.

In [ ]:
fig = hero(scores, meta, "machine_temperature")
fig.write_image(ROOT / "docs" / "hero.png", width=1400, height=560, scale=2)
fig

In [ ]:
for name in meta:
    small_multiples(scores, meta, name).show()

## Multivariate: SKAB

33 experiments on a water-circulation testbed with eight sensors sampled once per second (`other/2` is excluded because its anomaly
starts at point 104, before the training part ends). Following the SKAB convention, each experiment is fit on its first 300 points,
the threshold is the 99% quantile of points 300-400, and evaluation starts at point 400.

| Detector | Score | Contribution of sensor *i* |
|---|---|---|
| `zscore` | sum of squared standardized values | its own squared z-value (ignores correlations) |
| `pca` | combined T² + SPE index | complete decomposition contribution (Alcala & Qin 2011) |
| `autoencoder` | reconstruction error of a multichannel conv autoencoder | reconstruction error of channel *i* |

Before scoring, a causal moving average (EWMA, span 60 s) is subtracted from every sensor, so the detectors see deviations from the recent level instead of the slow drift of the testbed; the leave-one-out optimization below selected this for every detector. All scores are averaged over a trailing window of 20 s, and contributions always sum to the score.

Each experiment holds one long anomaly that covers about half of the test part. Two reference values help with reading the table:
- `chance_f1` comes from random time shifts of the alarms, pooled over all experiments.
- `point_f1 always-alarm` is the point-wise F1 of a detector that alarms on every test point. SKAB's own leaderboard reports point-wise F1.

`active_at_onset` is the share of experiments where the alarm was already on just before the anomaly started. In those cases the
delay of 0 s does not reflect real detection.

In [ ]:
skab = pd.read_csv(RESULTS / "skab_metrics.csv")
skab_pooled = pd.read_csv(RESULTS / "skab_pooled.csv")
skab_meta = json.loads((RESULTS / "skab_meta.json").read_text())
summary = skab_pooled.groupby("detector")[["tp", "fp", "events", "f1", "chance_f1", "p_value"]].mean()
summary = summary.join(skab.groupby("detector").agg(
    point_f1=("point_f1", "mean"), delay_s=("delay_s", "median"), active_at_onset=("active_at_onset", "mean"),
))
summary["point_f1 always-alarm"] = np.mean([f["point_f1_always"] for f in skab_meta["files"].values()])
summary.round(2)

### Which sensor triggered the alarm?

The explanation for an event is the mean sensor contribution over all alarms inside the labeled window. It is checked in two ways.

**Injected faults.** On the anomaly-free SKAB recording, a step or a linear drift of 3σ or 6σ is added to one sensor for 200 s,
with the same local fit and threshold protocol as above. `top1_accuracy` is the share of detected faults where the top sensor is the
one that was shifted; guessing gives 1/8. `alarm_without_fault` is how often the same stretch alarms without any injection, which
shows how much of the detection is caused by drift in the data.

In [ ]:
inj = pd.read_csv(RESULTS / "skab_injection.csv")
hit = inj[inj["detected"]]
keys = ["detector", "kind", "mag"]
inj.groupby(keys).agg(alarm_without_fault=("alarm_without_fault", "mean"), detected=("detected", "mean")).join(
    hit.groupby(keys).agg(top1_accuracy=("correct", "mean"))
).round(2)

In [ ]:
hit.pivot_table(index="sensor", columns="detector", values="correct", aggfunc="mean").round(2)

**Plausibility on real faults.** SKAB documents the fault type of each experiment, but not which sensor it affects. The expected
sensors below are a domain assumption: valve and leak experiments should show up in flow or pressure, rotor imbalance in the
accelerometers, hot water in the fluid temperature. Cavitation can affect hydraulics, vibration and motor current.

In [ ]:
attr = pd.read_csv(RESULTS / "skab_attribution.csv")
attr.groupby(["cause", "detector"])["plausible"].mean().unstack().round(2)

In [ ]:
attribution_heatmap(attr[attr["seed"] == 0], "autoencoder", SENSORS)

## Streaming

The same detectors run point by point with a buffer of w + 1 samples (`src/streaming.py`). Because every score depends only on the
trailing window, the streamed scores match the batch scores up to float rounding; the tests check this for every detector.

![streaming detection](streaming.gif)

Top: anomaly score against the threshold. Middle: share of the score per sensor. Right: the current explanation.

In [ ]:
pd.Series(json.loads((RESULTS / "streaming.json").read_text()))

## Optimization (leave-one-out)

Every setting is chosen on all *other* series and evaluated on the held-out one, so the numbers below are out-of-sample.
The grid covers the threshold quantile, EWMA smoothing of the score (0, ¼ or ½ of the window) and a minimum alarm duration
(0, 0.1 or ¼ of the window). Further options were added in stages:

- **A** post-processing only; SKAB uses the 60 s detrending already
- **B** ensembles over any subset of detectors (scores calibrated on normal validation data, then averaged)
- **C** context length: 0.5×, 1× or 2× the default for NAB, 10, 20 or 40 s for SKAB. Event merging and the post-window grace keep the default length, so the evaluation itself does not change
- **D** (SKAB only) fitting on the separate anomaly-free recording instead of the first 300 points of each experiment

Here the detector itself is part of the setting, which avoids picking the best detector after looking at the results.
`diff_vs_first` is the paired mean difference to stage A over series, with a bootstrap 95% interval.

In [ ]:
pd.concat({ds: pd.read_csv(RESULTS / f"opt_{ds}_auto_summary.csv").set_index("stage") for ds in ("nab", "skab")}).round(3)

On NAB, adding the context length gives a clear and stable gain: nearly every fold picks the same ensemble. On SKAB none of
the stages beats A. Shorter windows look better on the full grid but do not carry over to held-out experiments, and models fit on
the anomaly-free recording do worse than local fits even with their best setting. The pipeline therefore uses stage C for NAB
and stage A for SKAB (`src/config.py`).

Per detector, for the selected stage (`default` is the untuned setting, chance F1 and p-value are pooled over all series):

In [ ]:
from src.config import SELECTED_STAGE


def opt_table(ds):
    stage = SELECTED_STAGE[ds]
    loo = pd.read_csv(RESULTS / f"opt_{ds}_loo.csv").query("stage == @stage")
    pool = pd.read_csv(RESULTS / f"opt_{ds}_pooled.csv").query("stage == @stage").set_index(["detector", "setting"])
    t = loo.groupby(["detector", "setting"]).agg(
        mean_f1=("f1", "mean"), tp=("tp", "sum"), fp=("fp", "sum"), events=("events", "sum"),
    ).join(pool[["pooled_f1", "pooled_chance_f1", "pooled_p"]])
    return t.unstack("setting").round(3)


opt_table("nab")

In [ ]:
opt_table("skab")

Most frequently chosen setting per detector across the leave-one-out folds of the selected stage:

In [ ]:
def chosen(ds):
    loo = pd.read_csv(RESULTS / f"opt_{ds}_loo.csv")
    loo = loo[(loo["stage"] == SELECTED_STAGE[ds]) & (loo["setting"] == "tuned")]
    return loo.groupby("detector")[["variant", "q", "span", "k"]].agg(lambda c: c.value_counts().index[0])


pd.concat({ds: chosen(ds) for ds in ("nab", "skab")})